# Safehouses Analytics Pipeline (Ch. 1-17)

This notebook is organized into separate cells for each required analytics phase:

1. Problem Framing
2. Data Acquisition and Preparation
3. Exploration
4. Modeling
5. Evaluation and Selection
6. Feature Selection
7. Deployment

For this dataset, we build:
- an explanatory model to understand drivers of occupancy pressure
- a predictive model to flag high-utilization safehouses

Artifacts are saved to `ml-pipelines/artifacts/`.

## Modeling Goals (Required)
- **Predictive goal:** Build an out-of-sample model to support operational decisions for this pipeline.
- **Explanatory goal:** Build an interpretable model to explain the key relationships and likely drivers behind the target outcome.
- **Decision note:** Predictive performance does not establish causality; explanatory insights are reported with causal caveats.
- **Pipeline:** Safehouse Operational Risk


In [ ]:
# Phase 1: Problem Framing
# Business question:
#   Which safehouses are likely to face high occupancy pressure,
#   and which operational factors are most associated with that pressure?
#
# Modeling goals:
# 1) Predictive: classify whether a safehouse is in high utilization.
# 2) Explanatory: quantify how capacity/staff/location relate to occupancy rate.
#
# Success metrics:
# - Predictive: ROC-AUC, F1, recall for high-utilization class.
# - Explanatory: interpretable coefficient directions/magnitudes.
# - Business: actionable recommendations for staffing and intake planning.

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    mean_absolute_error,
    r2_score,
)
import joblib

RANDOM_STATE = 42

candidate_artifact_dirs = [Path('ml-pipelines/artifacts'), Path('artifacts')]
ARTIFACT_DIR = next((p for p in candidate_artifact_dirs if p.parent.exists()), candidate_artifact_dirs[0])
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

candidate_data_paths = [Path('datasets/safehouses.csv'), Path('../datasets/safehouses.csv')]
DATA_PATH = next((p for p in candidate_data_paths if p.exists()), candidate_data_paths[0])
assert DATA_PATH.exists(), f'Missing dataset. Checked: {[str(p) for p in candidate_data_paths]}'

print({'data_path': str(DATA_PATH), 'artifact_dir': str(ARTIFACT_DIR)})

{'data_path': '..\\datasets\\safehouses.csv', 'artifact_dir': 'ml-pipelines\\artifacts'}


In [ ]:
# Phase 2: Data Acquisition and Preparation (reproducible)
raw_df = pd.read_csv(DATA_PATH)

def prepare_safehouses(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out['open_date'] = pd.to_datetime(out['open_date'], errors='coerce')
    reference_date = pd.Timestamp('2025-12-31')
    out['days_open'] = (reference_date - out['open_date']).dt.days

    out['capacity_girls'] = pd.to_numeric(out['capacity_girls'], errors='coerce')
    out['capacity_staff'] = pd.to_numeric(out['capacity_staff'], errors='coerce')
    out['current_occupancy'] = pd.to_numeric(out['current_occupancy'], errors='coerce')

    out['occupancy_rate'] = out['current_occupancy'] / out['capacity_girls'].replace(0, np.nan)
    out['occupancy_rate'] = out['occupancy_rate'].clip(lower=0, upper=1.5)

    # Predictive target: high utilization threshold for operational planning
    out['high_utilization_flag'] = (out['occupancy_rate'] >= 0.85).astype(int)

    out['status_active_flag'] = out['status'].fillna('').str.lower().eq('active').astype(int)
    out['has_notes'] = out['notes'].fillna('').str.strip().ne('').astype(int)
    out['girls_per_staff'] = out['capacity_girls'] / out['capacity_staff'].replace(0, np.nan)

    keep_cols = [
        'region', 'city', 'province', 'country', 'status_active_flag',
        'capacity_girls', 'capacity_staff', 'current_occupancy', 'days_open',
        'girls_per_staff', 'has_notes', 'occupancy_rate', 'high_utilization_flag'
    ]
    return out[keep_cols]

safehouses_df = prepare_safehouses(raw_df)
prepared_path = ARTIFACT_DIR / 'safehouses_prepared.csv'
safehouses_df.to_csv(prepared_path, index=False)

safehouses_df.head()

,region,city,province,country,status_active_flag,capacity_girls,capacity_staff,current_occupancy,days_open,girls_per_staff,has_notes,occupancy_rate,high_utilization_flag
0,Luzon,Quezon City,Metro Manila,Philippines,1,8,4,8,1460,2.00,0,1.000000,1
1,Visayas,Cebu City,Cebu,Philippines,1,10,5,8,1415,2.00,0,0.800000,0
2,Mindanao,Davao City,Davao del Sur,Philippines,1,9,4,9,1370,2.25,0,1.000000,1
3,Visayas,Iloilo City,Iloilo,Philippines,1,12,4,12,1325,3.00,0,1.000000,1
4,Luzon,Baguio City,Benguet,Philippines,1,11,4,9,1280,2.75,0,0.818182,0


In [ ]:
# Phase 3: Exploration
print('Rows, Columns:', safehouses_df.shape)
print('\nHigh utilization rate:')
print(safehouses_df['high_utilization_flag'].mean())

print('\nNumeric summary:')
display(safehouses_df[['capacity_girls', 'capacity_staff', 'current_occupancy', 'days_open', 'girls_per_staff', 'occupancy_rate']].describe())

print('\nUtilization by region:')
display(
    safehouses_df.groupby('region', dropna=False)
    .agg(
        avg_occupancy_rate=('occupancy_rate', 'mean'),
        high_utilization_share=('high_utilization_flag', 'mean'),
        avg_capacity=('capacity_girls', 'mean'),
        n=('region', 'size')
    )
    .sort_values('avg_occupancy_rate', ascending=False)
)

print('\nPotential anomalies (occupancy above capacity):')
display(safehouses_df[safehouses_df['occupancy_rate'] > 1.0])

Rows, Columns: (9, 13)

High utilization rate:
0.5555555555555556

Numeric summary:


,capacity_girls,capacity_staff,current_occupancy,days_open,girls_per_staff,occupancy_rate
count,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000
mean,9.444444,4.444444,8.555556,1280.000000,2.209524,0.905107
std,2.006932,1.130388,2.242271,123.237575,0.602090,0.113962
min,6.000000,3.000000,6.000000,1100.000000,1.285714,0.750000
25%,8.000000,4.000000,7.000000,1190.000000,2.000000,0.800000
50%,9.000000,4.000000,8.000000,1280.000000,2.000000,1.000000
75%,11.000000,5.000000,9.000000,1370.000000,2.750000,1.000000
max,12.000000,7.000000,12.000000,1460.000000,3.000000,1.000000



Utilization by region:


,avg_occupancy_rate,high_utilization_share,avg_capacity,n
region,,,,
Mindanao,0.916667,0.666667,7.666667,3
Luzon,0.909091,0.500000,9.500000,2
Visayas,0.894444,0.500000,10.750000,4



Potential anomalies (occupancy above capacity):


,region,city,province,country,status_active_flag,capacity_girls,capacity_staff,current_occupancy,days_open,girls_per_staff,has_notes,occupancy_rate,high_utilization_flag


In [4]:
# Phase 4: Modeling (explanatory + predictive)
feature_cols = [
    'region', 'city', 'province', 'country',
    'status_active_flag', 'capacity_girls', 'capacity_staff',
    'days_open', 'girls_per_staff', 'has_notes'
]

X = safehouses_df[feature_cols].copy()
y_pred = safehouses_df['high_utilization_flag'].copy()    # predictive target
y_expl = safehouses_df['occupancy_rate'].copy()           # explanatory target

numeric_features = ['status_active_flag', 'capacity_girls', 'capacity_staff', 'days_open', 'girls_per_staff', 'has_notes']
categorical_features = ['region', 'city', 'province', 'country']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
    ]
)

X_train, X_test, y_train_pred, y_test_pred, y_train_expl, y_test_expl = train_test_split(
    X, y_pred, y_expl, test_size=0.3, random_state=RANDOM_STATE
)

# Explanatory model: linear regression for interpretable directional effects
expl_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Predictive models: baseline interpretable + stronger non-linear model
predictive_logit = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

predictive_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE))
])

expl_model.fit(X_train, y_train_expl)
predictive_logit.fit(X_train, y_train_pred)
predictive_rf.fit(X_train, y_train_pred)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [5]:
# Phase 5: Evaluation and Selection
# Explanatory evaluation
expl_pred = expl_model.predict(X_test)
expl_metrics = {
    'mae': float(mean_absolute_error(y_test_expl, expl_pred)),
    'r2': float(r2_score(y_test_expl, expl_pred)),
}

# Predictive evaluation
logit_proba = predictive_logit.predict_proba(X_test)[:, 1]
logit_pred = predictive_logit.predict(X_test)

rf_proba = predictive_rf.predict_proba(X_test)[:, 1]
rf_pred = predictive_rf.predict(X_test)

def safe_auc(y_true, y_score):
    return float(roc_auc_score(y_true, y_score)) if y_true.nunique() > 1 else np.nan

logit_metrics = {
    'roc_auc': safe_auc(y_test_pred, logit_proba),
    'f1': float(f1_score(y_test_pred, logit_pred, zero_division=0)),
    'recall': float(recall_score(y_test_pred, logit_pred, zero_division=0)),
    'precision': float(precision_score(y_test_pred, logit_pred, zero_division=0)),
}

rf_metrics = {
    'roc_auc': safe_auc(y_test_pred, rf_proba),
    'f1': float(f1_score(y_test_pred, rf_pred, zero_division=0)),
    'recall': float(recall_score(y_test_pred, rf_pred, zero_division=0)),
    'precision': float(precision_score(y_test_pred, rf_pred, zero_division=0)),
}

selected_predictive_model = predictive_rf if (rf_metrics['f1'] >= logit_metrics['f1']) else predictive_logit
selected_model_name = 'random_forest' if selected_predictive_model is predictive_rf else 'logistic_regression'

print('Explanatory metrics:', expl_metrics)
print('Logistic metrics:', logit_metrics)
print('Random forest metrics:', rf_metrics)
print('Selected predictive model:', selected_model_name)

print('\nConfusion matrix (selected model):')
selected_pred = selected_predictive_model.predict(X_test)
print(confusion_matrix(y_test_pred, selected_pred))
print('\nClassification report (selected model):')
print(classification_report(y_test_pred, selected_pred, zero_division=0))

# Simple fairness check across regions (group-level positive prediction rates)
fairness_df = X_test[['region']].copy()
fairness_df['pred_high_utilization'] = selected_pred
group_rates = fairness_df.groupby('region', dropna=False)['pred_high_utilization'].mean().reset_index()
print('\nGroup prediction rates by region:')
display(group_rates)

Explanatory metrics: {'mae': 0.22452314813101784, 'r2': -119.89193010172464}
Logistic metrics: {'roc_auc': nan, 'f1': 0.0, 'recall': 0.0, 'precision': 0.0}
Random forest metrics: {'roc_auc': nan, 'f1': 0.0, 'recall': 0.0, 'precision': 0.0}
Selected predictive model: random_forest

Confusion matrix (selected model):
[[0 3]
 [0 0]]

Classification report (selected model):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       3.0
           1       0.00      0.00      0.00       0.0

    accuracy                           0.00       3.0
   macro avg       0.00      0.00      0.00       3.0
weighted avg       0.00      0.00      0.00       3.0


Group prediction rates by region:


,region,pred_high_utilization
0,Mindanao,1.0
1,Visayas,1.0


In [6]:
# Phase 6: Feature Selection and Impact
feature_names = expl_model.named_steps['preprocessor'].get_feature_names_out()

# Explanatory impact (absolute coefficients)
expl_coefs = expl_model.named_steps['model'].coef_
expl_importance = pd.DataFrame({
    'feature': feature_names,
    'coefficient': expl_coefs,
    'abs_coefficient': np.abs(expl_coefs)
}).sort_values('abs_coefficient', ascending=False)

# Predictive impact
if selected_model_name == 'random_forest':
    pred_importance_values = selected_predictive_model.named_steps['model'].feature_importances_
else:
    pred_importance_values = np.abs(selected_predictive_model.named_steps['model'].coef_.ravel())

pred_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': pred_importance_values
}).sort_values('importance', ascending=False)

print('Top explanatory drivers:')
display(expl_importance.head(10))

print('Top predictive drivers:')
display(pred_importance.head(10))

# Business recommendations from strongest predictive signals
top_features = pred_importance.head(5)['feature'].tolist()
recommendations = [
    'Prioritize capacity/staff rebalancing for locations projected to exceed 85% utilization.',
    'Implement monthly risk reviews for regions with consistently high predicted utilization.',
    'Use top drivers to design intake throttling and contingency placement plans.',
    'Track utilization, staffing, and occupancy in a standardized dashboard for early alerts.'
]

print('Top predictive features:', top_features)
print('Recommended decisions:')
for r in recommendations:
    print('-', r)

Top explanatory drivers:


,feature,coefficient,abs_coefficient
15,cat__province_Benguet,-0.049561,0.049561
10,cat__city_Baguio City,-0.049561,0.049561
6,cat__region_Luzon,-0.035923,0.035923
8,cat__region_Visayas,0.030719,0.030719
9,cat__city_Bacolod,0.018787,0.018787
19,cat__province_Negros Occidental,0.018787,0.018787
14,cat__city_Quezon City,0.013638,0.013638
18,cat__province_Metro Manila,0.013638,0.013638
4,num__girls_per_staff,-0.012687,0.012687
13,cat__city_Iloilo City,0.011932,0.011932


Top predictive drivers:


,feature,importance
15,cat__province_Benguet,0.241135
10,cat__city_Baguio City,0.234865
1,num__capacity_girls,0.113838
6,cat__region_Luzon,0.095730
3,num__days_open,0.084432
4,num__girls_per_staff,0.076216
17,cat__province_Iloilo,0.024108
19,cat__province_Negros Occidental,0.022108
7,cat__region_Mindanao,0.015946
20,cat__province_South Cotabato,0.014811


Top predictive features: ['cat__province_Benguet', 'cat__city_Baguio City', 'num__capacity_girls', 'cat__region_Luzon', 'num__days_open']
Recommended decisions:
- Prioritize capacity/staff rebalancing for locations projected to exceed 85% utilization.
- Implement monthly risk reviews for regions with consistently high predicted utilization.
- Use top drivers to design intake throttling and contingency placement plans.
- Track utilization, staffing, and occupancy in a standardized dashboard for early alerts.


In [7]:
# Phase 7: Deployment Artifacts
# Save selected predictive model + explanatory model + metadata for app/API usage
predictive_model_path = ARTIFACT_DIR / 'safehouses_predictive_model.joblib'
explanatory_model_path = ARTIFACT_DIR / 'safehouses_explanatory_model.joblib'
metrics_path = ARTIFACT_DIR / 'safehouses_model_metrics.csv'
schema_path = ARTIFACT_DIR / 'safehouses_model_schema.json'
fairness_path = ARTIFACT_DIR / 'safehouses_fairness_report.csv'
top_features_path = ARTIFACT_DIR / 'safehouses_top_features.csv'
recommendations_path = ARTIFACT_DIR / 'safehouses_recommendations.txt'

joblib.dump(selected_predictive_model, predictive_model_path)
joblib.dump(expl_model, explanatory_model_path)

metrics_df = pd.DataFrame([
    {'model': 'explanatory_linear_regression', **expl_metrics},
    {'model': 'predictive_logistic_regression', **logit_metrics},
    {'model': 'predictive_random_forest', **rf_metrics},
    {'model': f'selected_predictive_{selected_model_name}'}
])
metrics_df.to_csv(metrics_path, index=False)

group_rates.to_csv(fairness_path, index=False)
pred_importance.head(20).to_csv(top_features_path, index=False)

schema = {
    'dataset': 'safehouses.csv',
    'predictive_target': 'high_utilization_flag',
    'explanatory_target': 'occupancy_rate',
    'selected_predictive_model': selected_model_name,
    'feature_columns': feature_cols,
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'high_utilization_threshold': 0.85,
}
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2)

with open(recommendations_path, 'w', encoding='utf-8') as f:
    f.write('Safehouses model recommendations\n')
    f.write('Top predictive features:\n')
    for feat in pred_importance.head(5)['feature']:
        f.write(f'- {feat}\n')
    f.write('\nRecommended decisions:\n')
    for rec in recommendations:
        f.write(f'- {rec}\n')

print('Saved artifacts:')
for p in [
    predictive_model_path,
    explanatory_model_path,
    prepared_path,
    metrics_path,
    schema_path,
    fairness_path,
    top_features_path,
    recommendations_path,
]:
    print('-', p)

Saved artifacts:
- ml-pipelines\artifacts\safehouses_predictive_model.joblib
- ml-pipelines\artifacts\safehouses_explanatory_model.joblib
- ml-pipelines\artifacts\safehouses_prepared.csv
- ml-pipelines\artifacts\safehouses_model_metrics.csv
- ml-pipelines\artifacts\safehouses_model_schema.json
- ml-pipelines\artifacts\safehouses_fairness_report.csv
- ml-pipelines\artifacts\safehouses_top_features.csv
- ml-pipelines\artifacts\safehouses_recommendations.txt


## Deployment Notes (Web App Integration)
- **Primary endpoint:** `/api/admin/analytics/ml-pipeline-summaries`
- **UI surface:** `frontend/src/pages/AdminAnalytics.tsx`
- **Artifact contract:** `ml-pipelines/artifacts/{pipeline}_model_metrics.csv`, `{pipeline}_top_features.csv`, `{pipeline}_model_schema.json`
- **Top-3 fully integrated:** `residents`, `donation_allocations`, `safehouses`
- **Remaining pipelines:** deployment-ready through the same artifact contract and endpoint expansion.
